# 1. Splitting Data # 
In this section we clean up the data and then split it into training data and test data on an 80/20 split. We do this so we can import the same training and test data into multiple notebooks and after creating models we can then see how each model does at predicting the same test data, and then measure the performance with root mean squared error.

In this first cell we import the data and then fill in some missing data, as a large number of non-religious schools had missing values for school_religious_character_detailed, which needed to be changed to 'Does not apply' before we can try and fit any models with these rows included.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("school_level_master_age_split.csv") # Imports the data to the notebook
# We now add in for the rows where school_religious_character_detailed was missing
df['school_religious_character_detailed'] = ( 
    df['school_religious_character_detailed'].fillna('Does not apply')
)

df.head() # Shows what our data looks like

,school_urn,time_period,school_name,school_laestab,la_name,new_la_code,establishment_type_group,pcon_code,pcon_name,school_admission_type,...,ks4_first_language_other_than_english_attainment8_average,ks4_first_language_other_than_english_engmath_94_percent,ks4_first_language_other_than_english_engmath_95_percent,ks4_first_language_other_than_english_progress8_average,ks4_non_mobile_pupil_count,ks4_non_mobile_pupil_percent,ks4_non_mobile_attainment8_average,ks4_non_mobile_engmath_94_percent,ks4_non_mobile_engmath_95_percent,ks4_non_mobile_progress8_average
0,100544,202526,David Game College,2016000,City of London,E09000001,Independent schools,E14001172,Cities of London and Westminster,Admission type unclassified,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100001,202526,City of London School for Girls,2016005,City of London,E09000001,Independent schools,E14001172,Cities of London and Westminster,Admission type unclassified,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100003,202526,City of London School,2016007,City of London,E09000001,Independent schools,E14001172,Cities of London and Westminster,Admission type unclassified,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,137181,202526,The UCL Academy,2024000,Camden,E09000007,Sponsored academies,E14001265,Hampstead and Highgate,Non-selective schools in other areas,...,44.8,58.7,40.2,NaN,147.0,89.6,47.5,64.6,49.0,NaN
4,100049,202526,Haverstock School,2024104,Camden,E09000007,Community school,E14001290,Holborn and St Pancras,Non-selective schools in other areas,...,38.1,53.7,37.0,NaN,116.0,84.1,41.9,61.2,43.1,NaN


There are still a lot of NaN values here, but we are not too worried about this at the moment as most of these are for columns which show attainment anyway and therefore will be excluded from our models.

In [2]:
import statsmodels.api as sm
print(df.columns) # Checking what available columns there are to use
print(df.info()) # Provides info on how many schools have data available for each column

Index(['school_urn', 'time_period', 'school_name', 'school_laestab', 'la_name',
       'new_la_code', 'establishment_type_group', 'pcon_code', 'pcon_name',
       'school_admission_type', 'school_religious_character_detailed',
       'sex_policy', 'age_range', 'min_age', 'max_age', 'allks_pupil_count',
       'allks_boys_count', 'allks_girls_count', 'endks4_pupil_count',
       'sen_pupil_count', 'sen_pupil_percent', 'sen_with_ehcp_pupil_count',
       'sen_with_ehcp_pupil_percent', 'sen_no_ehcp_pupil_count',
       'sen_no_ehcp_pupil_percent', 'attainment8_average',
       'attainment8eng_average', 'attainment8mat_average',
       'attainment8ebacc_average', 'attainment8open_average', 'pupil_count',
       'engmath_94_percent', 'engmath_95_percent', 'engmath_97_percent',
       'engmath_94_total', 'engmath_95_total', 'engmath_97_total',
       'gcse_91_percent', 'gcse_entries_average', 'qual_entries_average',
       'sci_triple_entering_percent', 'lan_multiple_entering_percent',
     

We now separate our target variable, attainment8_average, and select pretty much all of the potential covariates which don't include information about academic performance.

In [3]:
y = df['attainment8_average'] # We select the target variable

covariates = ['time_period', 'la_name', # We create a list of potential covariates
       'establishment_type_group', 'pcon_name',
       'school_admission_type', 'school_religious_character_detailed',
       'sex_policy', 'min_age', 'max_age', 'allks_pupil_count',
       'endks4_pupil_count',
       'sen_pupil_count', 'sen_pupil_percent', 'sen_with_ehcp_pupil_count',
       'sen_with_ehcp_pupil_percent', 'sen_no_ehcp_pupil_count',
       'sen_no_ehcp_pupil_percent', 'pupil_count',
       'gcse_entries_average', 'qual_entries_average',
       'sci_triple_entering_percent', 'lan_multiple_entering_percent',
       'ks4_disadvantaged_pupil_count', 'ks4_disadvantaged_pupil_percent',
       'ks4_not_known_disadvantaged_pupil_count',
       'ks4_not_known_disadvantaged_pupil_percent',
       'ks4_first_language_other_than_english_pupil_count',
       'ks4_first_language_other_than_english_pupil_percent',
       'ks4_non_mobile_pupil_count', 'ks4_non_mobile_pupil_percent']

We now look at how many of each of these covariates have missing values in them.

In [4]:
missing_summary = pd.DataFrame({ # Creates a table with columns: Covariates, Missing Values, Missing %
    'Missing values': df[covariates + ['attainment8_average']].isna().sum(), # Checks how many are missing in total
    'Missing percentage': (
        df[covariates + ['attainment8_average']].isna().mean() * 100 # Checks how many are missing as a percentage
    ).round(1)
})

missing_summary = missing_summary.sort_values( # We then order this table to easily identify potentially problematic covariates
    by='Missing percentage',
    ascending=False # Sorts it from most missing values to least
)

print(missing_summary) # Displays the table

                                                    Missing values  \
sen_no_ehcp_pupil_count                                       7413   
sen_pupil_percent                                             7413   
sen_no_ehcp_pupil_percent                                     7413   
sen_with_ehcp_pupil_percent                                   7413   
sen_with_ehcp_pupil_count                                     7413   
sen_pupil_count                                               7413   
ks4_non_mobile_pupil_percent                                  5154   
ks4_first_language_other_than_english_pupil_per...            5145   
ks4_not_known_disadvantaged_pupil_percent                     5103   
ks4_disadvantaged_pupil_percent                               5103   
ks4_non_mobile_pupil_count                                    5007   
ks4_first_language_other_than_english_pupil_count             4998   
ks4_disadvantaged_pupil_count                                 4956   
ks4_not_known_disadv

We can now see that around 42.9% of schools do not have information on the number of schools with special educational needs. We could remove some of these columns in order to maximise the number of rows in our data, but we instead choose to keep these in and remove rows with missing values, as these covariates could provide interesting information. We can't just replace all the missing values with a 0 in a similar fashion to what we did with the missing religious information as it is likely not a fair assumption that none of these schools had students with special educational needs, but instead just that the schools don't report that data back to the government, or at least it isn't included in the public dataset we are using.

In [5]:
print("Rows before removing missing values:", len(df)) # We do this to make sure the rows are in fact removed

model_data_clean = df.dropna(subset=covariates) # Removes a row if it has a missing value for any of the potential covariates

print("Rows after removing missing values:", len(model_data_clean)) # Shows the remaining rows
print("Rows removed:", len(df) - len(model_data_clean)) # Shows how many were removed

Rows before removing missing values: 17264
Rows after removing missing values: 9842
Rows removed: 7422


This is good, a lot of the missing values seem to have been all coming from the same schools, so we are still left with over half of our initial set of rows, which should be enough to produce some meaningful analysis.

In [6]:
import numpy as np

print("Missing values per column:") # Here we do a sanity check to make sure none of the potential covariates have missing values
print(model_data_clean.isna().sum()[df.isna().sum() > 0]) # Counts missing values per column

Missing values per column:
allks_pupil_count                        0
allks_boys_count                       361
allks_girls_count                      187
endks4_pupil_count                       0
sen_pupil_count                          0
                                      ... 
ks4_non_mobile_pupil_percent             0
ks4_non_mobile_attainment8_average      16
ks4_non_mobile_engmath_94_percent       16
ks4_non_mobile_engmath_95_percent       16
ks4_non_mobile_progress8_average      6581
Length: 63, dtype: int64

Infinite values per column:
Series([], dtype: int64)


We now split up the data into test data and training data and export this split as .csv files, so we can use the same split in a number of notebooks.

In [7]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split( # Splits the data into test and training data
    model_data_clean, # Takes the cleaned up data
    test_size=0.20, # Gives an 80/20 split
    random_state=42, 
    shuffle=True
)

train_data.to_csv("attainment8_train.csv", index=False) # Saves both data sets as .csv files
test_data.to_csv("attainment8_test.csv", index=False)

print("Total observations:", len(model_data_clean)) 
print("Training observations:", len(train_data)) # Check we did the split right by looking at the number of rows in each data set
print("Testing observations:", len(test_data)) 

print("\nTraining target mean:")
print(train_data["attainment8_average"].mean()) # Check the mean value of our target variable in each data set to make sure its not too different

print("\nTesting target mean:")
print(test_data["attainment8_average"].mean())

Total observations: 9842
Training observations: 7873
Testing observations: 1969

Training target mean:
46.91934459545281

Testing target mean:
46.96043676993398


Finally, we just double check that the .csv files have correctly been exported.

In [8]:
import os

print("Training file exists:", os.path.exists("attainment8_train.csv"))
print("Testing file exists:", os.path.exists("attainment8_test.csv"))

Current working directory: /home/jovyan
Training file exists: True
Testing file exists: True
